In [ ]:
# @title Env configuration (downstream)
!pip install --upgrade --quiet netcdf4 xarray

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim
from datetime import datetime, timedelta
import netCDF4

import xarray as xr
import sys
import os
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from scipy.ndimage import binary_dilation

import random
import matplotlib.pyplot as plt
from typing import Optional, Sequence, Tuple

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Hyperparameters for data
gt = False
start_iso_list_type = 'readAllDownloaded'                                            # ['readAllDownloaded', 'forCaseStudy']

### Hyperparameters for surrogate models
loss_function_type = 'focal'                                                    # ['weighted_cross_entropy', 'focal', 'focal+dice']
prediction_goal = 'trajectory'                                                  # ['trajectory', 'candidate_node']
surrogate_model_name = "deeplabv3plus"                                          # ['deeplabv3plus', 'deeplabv3']
backbone_name = 'xception'                                                      # ['xception', 'resnet']
is_heatmap = 'with_heatmap2'                                                  # ['with_heatmap1', 'with_heatmap2', 'with_heatmap3', 'with_heatmap5', 'without_heatmap']

### Hyperparameters for adversarial attacks
adv_binary_heat_map = False                                                     # [True, False]
pure_adv_loss_type = 'focal'                                                         # ['cross_entropy', 'focal_distance_weighted', 'weighted_cross_entropy', 'weighted_focal', 'focal', 'focal_plus_tversky'， 'wce_plus_tversky', 'tversky', 'hinge_topk']
adv_reg_type = 'none'                                        # ['none', 'weighted_prob', 'weighted_dist_to_target', 'unweighted_dist_to_target', 'focal_distance_weighted']
adv_loss_type = pure_adv_loss_type + '_' + adv_reg_type
use_loss_mask = False                                                           # [True, False]
loss_mask_ratio = 0.5
batch_size = 200  # [76, 152, 304]
epsilon = 10.0
num_steps = 1000
# lr = 2 * epsilon / num_steps
lr = 0.01
pure_attack_method = 'AOA'                                                           # ['PGD', 'AOA','TAAOWP'] ['Adam', 'AdamW', 'NAdam', 'Adamax', 'AdaBelief', 'AdaBound']
grad_strategy = 'pure'                                               # ['pure', 'grad_weight', 'distance_weight']
attack_method = pure_attack_method + '_' + grad_strategy + '_' + str(num_steps)
adv_is_heatmap = 'without_heatmap'                                                # ['without_heatmap', 'with_heatmap1', 'with_heatmap2', 'with_heatmap3', 'with_heatmap5']
strict_mask = True
deg_resolution  = 0.0
norm_type = 'L2'                                                                # ['L1', 'L2']
channels = [0,2,3]
lambda_reg = 1.0

no_dilation_for_learning_reg = True
gaus_sigma_loss = 0.01
gaus_signa_reg = 0.01

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/My Drive/GraphCast')
data_dir = "/content/drive/My Drive/Climate-data/Graphcast_1deg"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if loss_function_type == 'focal' and prediction_goal == 'trajectory' and is_heatmap == 'with_heatmap5':
    best_output_stride = 3
elif loss_function_type == 'focal' and prediction_goal == 'trajectory' and is_heatmap == 'with_heatmap3':
    best_output_stride = 1
elif loss_function_type == 'focal' and prediction_goal == 'trajectory' and is_heatmap == 'with_heatmap2':
    best_output_stride = 2
elif loss_function_type == 'focal' and prediction_goal == 'trajectory' and is_heatmap == 'with_heatmap1':
    best_output_stride = 6
elif loss_function_type in ['weighted_cross_entropy', 'cross_entropy', 'focal+dice']:
    best_output_stride = 3

surrogate_model_path = '/content/drive/My Drive/TempestExtremes_surrogate'
os.chdir(surrogate_model_path)
repo_root = "/content/drive/My Drive/TempestExtremes_surrogate/DeepLabV3Plus-Pytorch-master"
os.chdir(repo_root)
sys.path.insert(0, repo_root)
import network.modeling

def fetch_model(output_stride):
    MODEL_KEY = f"{surrogate_model_name}_{backbone_name}"
    model = network.modeling.__dict__[MODEL_KEY](num_classes=2, output_stride=output_stride, pretrained_backbone=False)
    if hasattr(model.backbone, "conv1"):
        old = model.backbone.conv1
        new = nn.Conv2d(
            in_channels=4,
            out_channels=old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=(old.bias is not None)
        )
        with torch.no_grad():
            new.weight[:, :3] = old.weight
            new.weight[:, 3:4] = old.weight[:, :1]
            if old.bias is not None:
                new.bias.copy_(old.bias)
        model.backbone.conv1 = new
    return model

save_dir = "/content/drive/My Drive/Climate-data/Graphcast_1deg"
if is_heatmap == 'without_heatmap':
    model_filename = f"best_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{prediction_goal}.pt"
else:
    model_filename = f"best_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{prediction_goal}_{is_heatmap}.pt"
model_path = os.path.join(save_dir, model_filename)
model = fetch_model(output_stride = best_output_stride)
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

# ['2008-04-27T0000', '2010-10-12T0000', '2013-11-03T0000', '2018-09-20T0000', '2019-08-02T0000', '2020-10-26T0000', '2022-09-09T0000', '2017-08-17T0000', '2017-09-16T0000', '2011-08-21T0000']
case_study_start_iso_list = ['2008-04-27T0000', '2010-10-12T0000', '2013-11-03T0000', '2018-09-20T0000', '2019-08-02T0000', '2020-10-26T0000', '2022-09-09T0000', '2017-08-17T0000', '2017-09-16T0000', '2011-08-21T0000']
if start_iso_list_type == 'forCaseStudy':
    start_iso_list = case_study_start_iso_list
    n_steps = 40
elif start_iso_list_type == 'readAllDownloaded':
    threshold = datetime.strptime("2019-01-01T0000", "%Y-%m-%dT%H%M")
    start_iso_list = []
    for filename in os.listdir('/content/drive/My Drive/Climate-data/Graphcast_1deg/'):
        try:
            date_part = filename[:13]
            file_datetime = datetime.strptime(date_part, "%Y-%m-%dT%H%M")
            if file_datetime >= threshold:
                start_iso_list.append(filename)
        except ValueError:
            continue
    start_iso_list = [item for item in start_iso_list if item not in case_study_start_iso_list]
    n_steps = 14

start_iso_list = sorted(start_iso_list, reverse = True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.9 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# @title Read data (downstream)
!pip install -q basemap basemap-data-hires scipy

import scipy.ndimage
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import numpy as np

prefix = 'gt' if gt else 'graphcast'

if start_iso_list_type == 'forCaseStudy':
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_case_study_{prediction_goal}.csv"))['dates'].tolist()
    hat_Z = torch.load(os.path.join(data_dir, f"{prefix}_hat_Z_case_study.pt")).to(device)
    adv_hat_target = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_T_target_case_study_{prediction_goal}.pt"))
    hat_target = torch.load(os.path.join(data_dir, "graphcast_test_true_Y_case_study_trajectory.pt")) # original downstream prediction, X->Y
    surrogate_pred = torch.load(os.path.join(data_dir, model_filename))
elif start_iso_list_type == 'readAllDownloaded':
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_{prediction_goal}.csv"))['dates'].tolist()
    hat_Z = torch.load(os.path.join(data_dir, f"{prefix}_hat_Z.pt")).to(device)   # [1788, 4, 181, 36], upstream forecasts
    hat_target = torch.load(os.path.join(data_dir, f"{prefix}_hat_T_target_{prediction_goal}.pt"))  # torch.Size([684, 181, 360]), original downstream prediction
    adv_hat_target = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_T_target_{prediction_goal}.pt"))  # torch.Size([1788, 181, 360]), adversirial downstream prediction
    surrogate_pred = torch.load(os.path.join(data_dir, model_filename))

print('all_dates: ', len(all_dates))
print('hat_Z: ', hat_Z.shape)
print('hat_target: ', hat_target.shape)
print('adv_hat_target: ', adv_hat_target.shape)

# print(surrogate_pred)

### Get the mean and std for each of the four variables
## compared to those provided by graphcast websites:
## ['wind_speed_10m', 'elevation', 'thickness', 'mean_sea_level_pressure']
##    [NA, NA, 35278.15150399017, 100960.6314111418]
##    [NA, NA, 1746.531780898365, 1330.9428153554159]
var_names = ['wind_speed_10m', 'elevation', 'thickness', 'mean_sea_level_pressure']
mean_tensor = torch.tensor([6.3266e+00, 3.8384e+02, 3.5398e+04, 1.0093e+05]).to(device)
std_tensor = torch.tensor([3.7596,  856.8956, 1829.8486, 1372.3270]).to(device)

# Statistics of the deviation of the input weather variables used for the downstream task
for c in range(4):
    vmin = hat_Z[:,c,:,:].min().item()
    vmax = hat_Z[:,c,:,:].max().item()
    mu = mean_tensor[c].item()
    sigma = std_tensor[c].item()
    norm_min = (vmin - mu) / sigma
    norm_max = (vmax - mu) / sigma
    print(f"{var_names[c]} (Channel {c}): raw min={vmin:.3f}, raw max={vmax:.3f}, (min-mu)/sigma={norm_min:.3f}, (max-mu)/sigma={norm_max:.3f}")

### Reconstruct the adversarial target with heatmap
def build_gaussian_heatmap(masks, radius, sigma=None, clamp=True):
    if radius <= 0:
        return masks.float()
    if sigma is None:
        sigma = max(1.0, radius / 3.0)

    B, H, W = masks.shape
    y = torch.arange(-radius, radius + 1, device=masks.device, dtype=torch.float32)
    x = torch.arange(-radius, radius + 1, device=masks.device, dtype=torch.float32)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    kernel = torch.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))                     # to ensure the val at center is 1.0 precisely
    kernel = kernel / kernel.max()
    weight = kernel.unsqueeze(0).unsqueeze(0)                                   # (1,1,k,k)

    inp = masks.unsqueeze(1).float()                                            # (B,1,H,W)
    heat = F.conv2d(inp, weight, padding=radius).squeeze(1)
    if clamp:                                                                   # for satefy consideration
        heat = heat.clamp_(0.0, 1.0)

    return heat

def build_binary_heatmap(masks, radius):
    B, H, W = masks.shape
    kernel_size = 2 * radius + 1
    masks_ = masks.unsqueeze(1).float()
    dilated = F.max_pool2d(masks_, kernel_size=kernel_size, stride=1, padding=radius)
    return dilated.squeeze(1)

adv_hat_target_no_dilation = adv_hat_target
if adv_is_heatmap == 'without_heatmap':
    radius = 0
    adv_hat_target = adv_hat_target
else:
    digits = ''.join(ch for ch in str(adv_is_heatmap) if ch.isdigit())
    radius = int(digits) if digits else 1
    if adv_binary_heat_map == True:
        print(f"reconstructing adversarial targets (binary, r={radius})...")
        adv_hat_target = build_binary_heatmap(adv_hat_target, radius)
    elif adv_binary_heat_map == False:
        print(f"reconstructing adversarial targets (gaussian, r={radius})...")
        adv_hat_target = build_gaussian_heatmap(adv_hat_target, radius)
    print("completed")

hat_Z_norm = (hat_Z - mean_tensor.view(1, 4, 1, 1).to(device)) / std_tensor.view(1, 4, 1, 1)
hat_Z_norm = hat_Z_norm.to(device)

model.eval()
def batched_forward(model, x, surrogate_infer_batch_size):
    outs = []
    with torch.inference_mode():
        for i in range(0, x.size(0), batch_size):
            xb = x[i:i+batch_size]
            out = model(xb)                                                     # [B,C,H,W] -> logits
            outs.append(out)
    return torch.cat(outs, dim=0)
logits = batched_forward(model, hat_Z_norm, surrogate_infer_batch_size=batch_size)
org_prob_1 = torch.softmax(logits, dim=1)[:, 1]
org_hat_forecast = (org_prob_1 > 0.5).long()

org = org_hat_forecast.long()
adv = adv_hat_target.long()

total_org = org.numel()
ones_org = int((org == 1).sum().item())
zeros_org = int((org == 0).sum().item())

total_adv = adv.numel()
ones_adv = int((adv == 1).sum().item())
zeros_adv = int((adv == 0).sum().item())

print(f"org_hat_forecast → total = {total_org}, ones = {ones_org}, zeros = {zeros_org}")
print(f"adv_hat_target   → total = {total_adv}, ones = {ones_adv}, zeros = {zeros_adv}")


all_dates:  1788
hat_Z:  torch.Size([1788, 4, 181, 360])
hat_target:  torch.Size([1788, 181, 360])
adv_hat_target:  torch.Size([1788, 181, 360])
wind_speed_10m (Channel 0): raw min=0.000, raw max=36.277, (min-mu)/sigma=-1.683, (max-mu)/sigma=7.966
elevation (Channel 1): raw min=-141.383, raw max=5631.773, (min-mu)/sigma=-0.613, (max-mu)/sigma=6.124
thickness (Channel 2): raw min=31092.109, raw max=39242.746, (min-mu)/sigma=-2.353, (max-mu)/sigma=2.101
mean_sea_level_pressure (Channel 3): raw min=91865.984, raw max=106458.062, (min-mu)/sigma=-6.605, (max-mu)/sigma=4.028
org_hat_forecast → total = 116506080, ones = 39165, zeros = 116466915
adv_hat_target   → total = 116506080, ones = 2335, zeros = 116503745


In [ ]:
# @title Adversarial Attack methods (downstream)
import random
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Sequence, Tuple
import torch
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors

import torch
import torch.nn.functional as F
import numpy as np
import scipy.ndimage

!pip install adabelief-pytorch adabound
from adabelief_pytorch import AdaBelief
import adabound

from torch.nn.utils import clip_grad_norm_

def compute_distance_weight(targets, sigma, alpha=30.0, eps=1e-6):
    device = targets.device

    if targets.dim() == 4:
        targets = targets[:, 0]
    B, H, W = targets.shape
    lats = np.linspace(-90.0, 90.0, H)
    lons = np.linspace(0.0, 360.0, W, endpoint=False)
    lat_vals = torch.tensor(np.deg2rad(lats), dtype=torch.float32)
    lon_vals = torch.tensor(np.deg2rad(lons), dtype=torch.float32)
    lat_grid, lon_grid = torch.meshgrid(lat_vals, lon_vals, indexing='ij')
    lat_flat = lat_grid.to(device).reshape(-1)
    lon_flat = lon_grid.to(device).reshape(-1)
    distance_weight = []
    for b in range(B):
        tgt_mask = targets[b] > 0.5
        tgt_idx = tgt_mask.view(-1).nonzero(as_tuple=False).squeeze(1)
        if tgt_idx.numel() == 0:
            this_distance_weight = torch.zeros((H, W), device=device)
        else:
            tgt_lat = lat_flat[tgt_idx]
            tgt_lon = lon_flat[tgt_idx]
            cand_lat = lat_flat.unsqueeze(1)
            cand_lon = lon_flat.unsqueeze(1)
            dlon = cand_lon - tgt_lon
            cosD = cand_lat.sin() * tgt_lat.sin() + cand_lat.cos() * tgt_lat.cos() * dlon.cos()
            D_rad = cosD.clamp(-1, 1).acos()
            min_dist = D_rad.min(dim=1)[0].view(H, W)
            num_min_dist_zero = (min_dist == 0).sum().item()
            this_distance_weight = torch.exp(-(min_dist**2) / (2 * sigma**2))

            ## OPTION: Gaussian decay
            # distance_weight.append(torch.exp(-alpha * min_dist))

            ## OPTION: Exponential decay
            # mn, mx = min_dist.min(), min_dist.max()
            # norm_dist = (min_dist - mn) / (mx - mn + eps)
            # distance_weight.append(torch.exp(-alpha * norm_dist))

        distance_weight.append(this_distance_weight)

        # # CHECK THE CORRECTNESS BY PLOTTING OUT WHAT WE OBRAIN
        # print(f"Batch {b}: Number of pixels where min_dist == 0: {num_min_dist_zero}")
        # print("Sampled distance_weight at target points:", this_distance_weight[targets[b] > 0.5])
        # print("Max distance_weight:", this_distance_weight.max().item())
        # print("Min distance_weight:", this_distance_weight.min().item())
        # red_white = mcolors.LinearSegmentedColormap.from_list("red_white", ["white", "red"])
        # plt.figure(figsize=(8, 4))
        # plt.imshow(this_distance_weight.cpu().numpy(), cmap=red_white, origin='lower', vmin=0, vmax=1)
        # plt.colorbar(label='Distance Weight')
        # plt.title(f'Batch {b}: {tgt_idx.numel()} target pixels')
        # plt.xlabel('Grid X')
        # plt.ylabel('Grid Y')
        # plt.tight_layout()
        # plt.show()

    return torch.stack(distance_weight, dim=0)

class AdamSeries:
    def __init__(self, loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr, optimizer_name="Adam"):
        self.loss_scheduler = loss_scheduler
        self.reg_scheduler = reg_scheduler
        self.model = model
        self.epsilon = epsilon
        self.num_steps = num_steps
        self.device = device
        self.lr = lr
        self.optimizer_name = optimizer_name

    def __call__(self, hat_Z, org_forecasts, org_adv_targets, adv_targets, hat_targets):
        orig = hat_Z.to(self.device)
        delta = torch.zeros_like(orig, device=self.device, requires_grad=True)
        if self.optimizer_name == 'Adam':
            optimizer = optim.Adam([delta], lr=self.lr)
        elif self.optimizer_name == 'AdamW':
            optimizer = optim.AdamW([delta], lr=self.lr)
        elif self.optimizer_name == 'NAdam':
            optimizer = optim.NAdam([delta], lr=self.lr)
        elif self.optimizer_name == 'Adamax':
            optimizer = optim.Adamax([delta], lr=self.lr)
        elif self.optimizer_name == 'AdaBelief':
            optimizer = AdaBelief([delta], lr=self.lr)
        elif self.optimizer_name == 'AdaBound':
            optimizer = adabound.AdaBound([delta], lr=self.lr)
        else:
            raise ValueError(self.optimizer_name)
        best_loss = float('inf')
        best_delta = delta.detach().clone()
        batch_size = org_forecasts.shape[0]
        adv_targets_plus_org_forecasts = []
        for b in range(batch_size):
            org_surrogate = org_forecasts[b].detach().cpu().bool().numpy()
            if no_dilation_for_learning_reg:
                tgt = org_adv_targets[b].detach().cpu().bool().numpy()
            else:
                tgt = adv_targets[b].detach().cpu().bool().numpy()
            black_box = hat_targets[b].detach().cpu().bool().numpy()
            H, W = org_surrogate.shape
            mixed = (black_box | tgt).copy()
            labels, num = scipy.ndimage.label(org_surrogate)
            for n in range(1, num + 1):
                cl = (labels == n)
                if not (cl & mixed).any():
                    mixed |= cl
            adv_targets_plus_org_forecasts.append(mixed)
        adv_targets_plus_org_forecasts = torch.from_numpy(np.stack(adv_targets_plus_org_forecasts)).long().to(self.device)
        for t in range(self.num_steps):
            if delta.grad is not None:
                delta.grad.zero_()
            optimizer.zero_grad()
            logits = self.model(orig + delta)
            adv_loss = self.loss_scheduler(logits, adv_targets)
            reg_loss = self.reg_scheduler(delta, logits, adv_targets_plus_org_forecasts)
            loss = adv_loss + lambda_reg * reg_loss
            print(f"Step {t+1}: loss = {loss.item()}, adv_loss = {adv_loss.item()}, reg_loss = {reg_loss.item()}, "
                  f"pert_ch0 = {delta.detach().abs()[:,0].sum().item():.4f}, "
                  f"pert_ch2 = {delta.detach().abs()[:,2].sum().item():.4f}, "
                  f"pert_ch3 = {delta.detach().abs()[:,3].sum().item():.4f}, "
                  f"pert_sum = {delta.detach().abs()[:,[0,2,3]].sum().item():.4f}")
            if loss.item() < best_loss:
                best_loss, best_delta = loss.item(), delta.detach().clone()
            loss.backward()
            if grad_strategy == 'pure':
                pass
            elif grad_strategy == 'grad_weight':
                delta.grad.mul_(delta.grad.abs())
            elif grad_strategy == 'distance_weight':
                dw = compute_distance_weight(adv_targets_plus_org_forecasts, gaus_sigma_loss).unsqueeze(1)
                delta.grad.mul_(dw)
            optimizer.step()
            with torch.no_grad():
                delta.clamp_(-self.epsilon, self.epsilon)
                delta[:, 1, :, :] = 0.0
            # if (t+1) % 100 == 0 or t == 0:
            #     with torch.no_grad():
            #         plot_adv_and_delta(logits, adv_targets, delta, t)
        adv_hat_Z = hat_Z + best_delta
        return adv_hat_Z.cpu(), torch.argmax(self.model(adv_hat_Z), dim=1).cpu()

class AOA:
    def __init__(self, loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr=0.01, beta=0.5, grad_clip=1.0):
        self.loss_scheduler = loss_scheduler
        self.reg_scheduler = reg_scheduler
        self.model = model
        self.epsilon = float(epsilon)
        self.num_steps = int(num_steps)
        self.device = device
        self.lr = float(lr)
        self.beta = float(beta)
        self.grad_clip = float(grad_clip)
        self.adam_betas = (self.beta, 0.999)

    def __call__(self, hat_Z, org_forecasts, org_adv_targets, adv_targets, hat_targets):
        self.model.eval()
        orig = hat_Z.to(self.device).float()
        adv_targets = adv_targets.to(self.device)
        delta = torch.zeros_like(orig, device=self.device, requires_grad=True, dtype=torch.float32)

        optimizer = torch.optim.Adam([delta], lr=self.lr, betas=self.adam_betas)
        best_loss = float('inf')
        best_delta = torch.zeros_like(delta).detach().clone()

        for t in range(self.num_steps):
            optimizer.zero_grad()
            logits = self.model(orig + delta)
            loss = self.loss_scheduler(logits, adv_targets)

            if not torch.isfinite(loss):
                print(f"Step {t+1}: non-finite loss encountered -> {loss}")
                break

            if loss.item() < best_loss:
                best_loss = loss.item()
                best_delta = delta.detach().clone()

            with torch.no_grad():
                pert_ch0 = float(delta.detach().abs()[:, 0].sum().item()) if delta.shape[1] > 0 else 0.0
                pert_ch2 = float(delta.detach().abs()[:, 2].sum().item()) if delta.shape[1] > 2 else 0.0
                pert_ch3 = float(delta.detach().abs()[:, 3].sum().item()) if delta.shape[1] > 3 else 0.0
                idxs = [i for i in [0,2,3] if i < delta.shape[1]]
                pert_sum = float(delta.detach().abs()[:, idxs].sum().item()) if idxs else 0.0

            print(f"Step {t+1}: loss = {loss.item():.6f}, pert_ch0 = {pert_ch0:.4f}, pert_ch2 = {pert_ch2:.4f}, pert_ch3 = {pert_ch3:.4f}, pert_sum = {pert_sum:.4f}")

            loss.backward()

            if delta.grad is None:
                print(f"Step {t+1}: no gradient on delta, stopping")
                break

            clip_grad_norm_([delta], self.grad_clip)

            optimizer.step()

            with torch.no_grad():
                delta.clamp_(-self.epsilon, self.epsilon)
                if delta.shape[1] > 1:
                    delta[:, 1, :, :] = 0.0

        best_delta = best_delta.to(self.device)
        adv_hat_Z = (hat_Z.to(self.device).float() + best_delta).to(self.device)

        self.model.to(self.device)
        self.model.eval()
        with torch.no_grad():
            preds = torch.argmax(self.model(adv_hat_Z), dim=1).cpu()

        return adv_hat_Z.cpu(), preds
class PGD:
    def __init__(self, loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr):
        self.loss_scheduler = loss_scheduler
        self.reg_scheduler = reg_scheduler
        self.model = model
        self.epsilon = epsilon
        self.num_steps = num_steps
        self.device = device
        self.lr = lr

    def __call__(self, hat_Z, org_forecasts, org_adv_targets, adv_targets, hat_targets):
        # 'org_forecasts': 0/1 (considered dilation); 'org_adv_targets': 0/1; 'adv_targets': 0/1/0.6065/0.3679; 'hat_targets': 0/1
        # batch_size = org_forecasts.shape[0]
        # for b in range(batch_size):
        #     print(org_forecasts[b].sum().item(),
        #         org_adv_targets[b].sum().item(),
        #         hat_targets[b].sum().item())

        orig = hat_Z.to(self.device)
        delta = torch.zeros_like(orig, device=self.device, requires_grad=True)
        best_loss = float('inf')
        best_delta = delta.detach().clone()

        batch_size = org_forecasts.shape[0]
        adv_targets_plus_org_forecasts = []

        for b in range(batch_size):
            org_black_box = hat_targets[b].cpu().bool().numpy()
            org_surrogate_model = org_forecasts[b].cpu().bool().numpy()
            if no_dilation_for_learning_reg == True:
                tgt = org_adv_targets[b].cpu().bool().numpy()
            elif no_dilation_for_learning_reg == False:
                tgt = adv_targets[b].cpu().bool().numpy()

            org_surrogate_model_label, org_surrogate_model_num = scipy.ndimage.label(org_surrogate_model) # num_features: the total number of connected regions

            # H, W = org_surrogate_model.shape
            # org_surrogate_model_center = np.zeros((H, W), dtype=bool)
            # for n in range(1, org_surrogate_model_num + 1):
            #     ys, xs = np.where(org_surrogate_model_label == n)
            #     if ys.size == 0:
            #         continue
            #     yc, xc = ys.mean(), xs.mean()
            #     idx = np.argmin((ys - yc) ** 2 + (xs - xc) ** 2)
            #     y0, x0 = int(ys[idx]), int(xs[idx])
            #     org_surrogate_model_center[y0, x0] = True
            # mixed = (org_black_box | tgt | org_surrogate_model_center)          # need the initial surrogate forecast, otherwise, the extra loss can not be reduced and will influence the overall performance

            H, W = org_surrogate_model.shape
            mixed = (org_black_box | tgt).copy()
            for n in range(1, org_surrogate_model_num + 1):
                cluster_mask = (org_surrogate_model_label == n)
                if not (cluster_mask & mixed).any():
                    mixed |= cluster_mask

            # print(f"Batch {b}: number of True = {mixed.sum()}")
            adv_targets_plus_org_forecasts.append(mixed)

        adv_targets_plus_org_forecasts = torch.from_numpy(np.stack(adv_targets_plus_org_forecasts)).long().to(self.device)
        # for b in range(adv_targets_plus_org_forecasts.shape[0]):
        #     count = adv_targets_plus_org_forecasts[b].sum().item()
        #     print(f"Batch {b}: number of 1s = {count}")

        with torch.no_grad():
            init_logits = self.model(orig)
            p1_prime = torch.softmax(init_logits, dim=1)[:, 1]                  # fetch the probability of being classified as 1 (i.e., trajactory node)

            hat_t = hat_targets.to(self.device)
            adv_t = org_adv_targets.to(self.device)
            org_f = org_forecasts.to(self.device)

            m_correct_surrogate_pred  = (adv_t != hat_t) & (org_f != adv_t)   # correct initial forecasts by the surrogate model
            m_incorrect_surrogate_pred = (adv_t != hat_t) & (org_f == adv_t)  # incorrect initial forecasts by the surrogate model

            mask_changed = m_incorrect_surrogate_pred.float()

        for t in range(self.num_steps):
            if delta.grad is not None:
                delta.grad.zero_()
            adv_inputs = orig + delta
            logits = self.model(adv_inputs)

            adv_loss = self.loss_scheduler(logits, adv_targets, p1_prime=p1_prime, mask_changed=mask_changed)
            reg_loss = self.reg_scheduler(delta, logits, adv_targets_plus_org_forecasts)
            loss = adv_loss + lambda_reg * reg_loss

            print(f"Step {t+1}: loss = {loss.item()}, adv_loss = {adv_loss.item()}, reg_loss = {reg_loss.item()}, "
                  f"pert_ch0 = {delta.detach().abs()[:,0].sum().item():.4f}, "
                  f"pert_ch2 = {delta.detach().abs()[:,2].sum().item():.4f}, "
                  f"pert_ch3 = {delta.detach().abs()[:,3].sum().item():.4f}, "
                  f"pert_sum = {delta.detach().abs()[:,[0,2,3]].sum().item():.4f}")

            if loss.item() < best_loss:
                best_loss, best_delta = loss.item(), delta.detach().clone()
            loss.backward()

            with torch.no_grad():
                # UPDATING STRATEGY ONE: PURE PGD
                if grad_strategy == 'pure':
                    delta -= self.lr * delta.grad.sign()
                # UPDATING STRATEGY TWO: GRADIENT-BASED PGD
                elif grad_strategy == 'grad_weight':
                    delta -= self.lr * delta.grad.sign() * delta.grad.abs()
                # UPDATING STRATEGY THREE: DISTANCE-BASED PGD
                elif grad_strategy == 'distance_weight':
                    distance_weight = compute_distance_weight(adv_targets_plus_org_forecasts, gaus_sigma_loss)
                    distance_weight = distance_weight.unsqueeze(1)
                    delta -= self.lr * (distance_weight) * delta.grad.sign()

                delta.clamp_(-self.epsilon, self.epsilon)
                delta[:, 1, :, :] = 0.0

            # if (t+1) % 500 == 0 or t == 0:
            #     with torch.no_grad():
            #         plot_adv_and_delta(logits, adv_targets, delta, t)

        adv_hat_Z = hat_Z + best_delta
        return adv_hat_Z.cpu(), torch.argmax(self.model(adv_hat_Z), dim=1).cpu()

def plot_adv_and_delta(logits, targets, delta, step):
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

    softmax = torch.nn.functional.softmax(logits, dim=1)
    prob_1 = softmax[:, 1].detach().cpu().numpy()
    delta_np = delta.detach().cpu().numpy()
    delta_grad = None
    if delta.grad is not None:
        delta_grad = delta.grad.detach().cpu().numpy()

    indices = np.arange(min(5, prob_1.shape[0]))
    delta_vars = [0, 2, 3]
    delta_names = [
        'Perturbation: 10m Wind Speed',
        'Perturbation: Geo Thickness',
        'Perturbation: MSLP'
    ]
    grad_names = [
        'Gradient Mag: 10m Wind',
        'Gradient Mag: Geo Thick',
        'Gradient Mag: MSLP'
    ]
    white_red = LinearSegmentedColormap.from_list("white_red", ["white", "red"])
    marker_size = 5

    fig, axes = plt.subplots(
        nrows=len(indices), ncols=8,
        figsize=(32, 6 * len(indices)),
        gridspec_kw={'width_ratios': [1]*8, 'wspace': 0.3}
    )

    for i, idx in enumerate(indices):
        ax_row = axes[i] if len(indices) > 1 else axes

        # column 0: forecast
        arr = prob_1[idx]
        mask = arr > 0.5
        y_f, x_f = np.where(mask)
        c_f = arr[y_f, x_f]
        sc0 = ax_row[0].scatter(x_f, y_f, c=c_f, cmap='Reds', s=marker_size, vmin=0.5, vmax=1.0)
        ax_row[0].set_title('Forecast P>0.5')
        ax_row[0].set_xlim([0, arr.shape[1]])
        ax_row[0].set_ylim([arr.shape[0], 0])
        ax_row[0].set_aspect('equal')
        cb0 = plt.colorbar(sc0, ax=ax_row[0], fraction=0.04, pad=0.01)
        cb0.ax.tick_params(labelsize=6)

        # column 1: target
        if targets.dim() == 4:
            target_np = targets[idx, 0].detach().cpu().numpy()
        else:
            target_np = targets[idx].detach().cpu().numpy()
        # mask = target_np == 1
        mask = target_np > 0
        y_t, x_t = np.where(mask)
        c_t = target_np[y_t, x_t]
        sc1 = ax_row[1].scatter(x_t, y_t, c=c_t, cmap='Reds', s=marker_size, vmin=0, vmax=1)
        ax_row[1].set_title('Adversarial Target')
        ax_row[1].set_xlim([0, target_np.shape[1]])
        ax_row[1].set_ylim([target_np.shape[0], 0])
        ax_row[1].set_aspect('equal')
        cb1 = plt.colorbar(sc1, ax=ax_row[1], fraction=0.04, pad=0.01)
        cb1.ax.tick_params(labelsize=6)

        # columns 2–4: perturbations
        for j, d in enumerate(delta_vars):
            raw = delta_np[idx, d]
            mask = np.abs(raw) > 0.01 * np.max(np.abs(raw))
            y, x = np.where(mask)
            c = raw[y, x]
            sc = ax_row[j+2].scatter(x, y, c=c, cmap='bwr', s=marker_size, vmin=raw.min(), vmax=raw.max())
            ax_row[j+2].set_title(delta_names[j])
            ax_row[j+2].set_aspect('equal')
            ax_row[j+2].set_xlim([0, raw.shape[1]])
            ax_row[j+2].set_ylim([raw.shape[0], 0])
            cb = plt.colorbar(sc, ax=ax_row[j+2], fraction=0.04, pad=0.01)
            cb.ax.tick_params(labelsize=6)

        # columns 5–7: gradient magnitudes
        if delta_grad is not None:
            for k, ch in enumerate(delta_vars):
                grad_map = np.abs(delta_grad[idx, ch])
                mask = grad_map > 0.01 * grad_map.max()
                y, x = np.where(mask)
                c = grad_map[y, x]
                sc = ax_row[5+k].scatter(x, y, c=c, cmap=white_red, s=marker_size, vmin=grad_map.min(), vmax=grad_map.max())
                ax_row[5+k].set_title(grad_names[k])
                ax_row[5+k].set_aspect('equal')
                ax_row[5+k].set_xlim([0, grad_map.shape[1]])
                ax_row[5+k].set_ylim([grad_map.shape[0], 0])
                cb = plt.colorbar(sc, ax=ax_row[5+k], fraction=0.04, pad=0.01)
                cb.ax.tick_params(labelsize=6)

        for ax in ax_row:
            ax.set_xlabel('Lon')
            ax.set_ylabel('Lat')
            ax.set_facecolor('white')

    plt.suptitle(f'Adversarial & Delta Visualization (Iter {step+1})', fontsize=16)
    plt.subplots_adjust(left=0.05, right=0.96, top=0.90, bottom=0.05)
    plt.show()


In [ ]:
# @title class::RegScheduler
class RegScheduler:
    def __init__(self, reg_type, lat_grid, lon_flat):
        self.reg_type = reg_type
        self.channels = channels
        self.norm_type = norm_type
        self.lat_flat = lat_grid.reshape(-1).to(device)
        self.lon_flat = lon_grid.reshape(-1).to(device)

    def __call__(self, delta, logits, adv_targets_plus_org_forecasts):
        if self.reg_type == "weighted_prob":
            reg_loss = self.reg_loss_weighted_prob(delta, logits, power = 2.0)
        elif self.reg_type == "weighted_dist_to_target":
            reg_loss = self.reg_loss_weighted_dist_to_target(delta, adv_targets_plus_org_forecasts)
        elif self.reg_type == "unweighted_dist_to_target":
            reg_loss = self.reg_loss_unweighted_dist_to_target(delta, adv_targets_plus_org_forecasts)
        elif self.reg_type == "none":
            reg_loss = torch.tensor(0.0)
        else:
            raise ValueError(f"Unknown reg_type: {self.reg_type}")
        return reg_loss

    def norm(self, x):
        if self.norm_type == 'L1':
            return torch.abs(x)
        elif self.norm_type == 'L2':
            return x**2
        else:
            raise ValueError(f"Unknown norm_type: {self.norm_type}")

    def reg_loss_weighted_dist_to_target(self, delta, targets):
        B, _, H, W = delta.shape
        weight = compute_distance_weight(targets, gaus_signa_reg)
        weight = 1.0 - weight

        # for b in range(B):
        #     plt.figure(figsize=(4, 4))
        #     plt.imshow(weight[b].detach().cpu(), cmap='viridis', origin='lower')
        #     plt.title(f'Regularization Weight (sample {b})')
        #     plt.colorbar(label='weight')
        #     plt.xlabel('Longitude')
        #     plt.ylabel('Latitude')
        #     plt.show()

        reg_terms = []
        for b in range(B):
            for ch in self.channels:
                reg_terms.append(self.norm(weight[b] * delta[b, ch]).mean())
        return torch.stack(reg_terms).mean()

    def reg_loss_weighted_prob(self, delta, logits, power=1.0):
        prob_1 = torch.softmax(logits, dim=1)[:, 1]
        reg_terms = []
        for ch in self.channels:
            weight = 1.0 - prob_1 ** power
            reg_terms.append(self.norm(weight * delta[:, ch]).mean())
        reg_loss = torch.stack(reg_terms).mean()
        return reg_loss

    def reg_loss_unweighted_dist_to_target(self, delta, targets):
        dist_thr = deg_resolution * np.pi / 180.0
        device = delta.device
        B, _, H, W = delta.shape
        if targets.dim() == 4:
            targets = targets[:, 0]
        lats = np.linspace(-90.0, 90.0, H)
        lons = np.linspace(0.0, 360.0, W, endpoint=False)
        lat_vals = torch.tensor(np.deg2rad(lats), dtype=torch.float32, device=device)
        lon_vals = torch.tensor(np.deg2rad(lons), dtype=torch.float32, device=device)
        lat_grid, lon_grid = torch.meshgrid(lat_vals, lon_vals, indexing='ij')
        lat_flat = lat_grid.reshape(-1)
        lon_flat = lon_grid.reshape(-1)
        reg_terms = []
        for b in range(B):
            tgt_mask = targets[b] > 0.5
            tgt_idx = tgt_mask.view(-1).nonzero(as_tuple=False).squeeze(1)
            if tgt_idx.numel() == 0:
                min_dist = torch.zeros((H, W), device=device)
                mask_far = torch.zeros((H, W), device=device)
            else:
                tgt_lat = lat_flat[tgt_idx]
                tgt_lon = lon_flat[tgt_idx]
                cand_lat = lat_flat.unsqueeze(1)
                cand_lon = lon_flat.unsqueeze(1)
                dlon = cand_lon - tgt_lon
                cosD = cand_lat.sin() * tgt_lat.sin() + cand_lat.cos() * tgt_lat.cos() * dlon.cos()
                D_rad = cosD.clamp(-1, 1).acos()
                min_dist = D_rad.min(dim=1)[0].view(H, W)
                mask_far = (min_dist > dist_thr).float()

            # plt.figure(figsize=(8, 4))
            # plt.imshow(min_dist.cpu().numpy(), cmap='jet', origin='lower')
            # target_locs = (targets[b] > 0.5).nonzero(as_tuple=False)
            # if len(target_locs) > 0:
            #     ys = target_locs[:, 0].cpu().numpy()
            #     xs = target_locs[:, 1].cpu().numpy()
            #     plt.scatter(xs, ys, marker='o', s=40)
            # plt.colorbar(label='Min Dist (rad)')
            # plt.title(f'Batch {b}: Min Spherical Distance Map')
            # plt.xlabel('Grid X')
            # plt.ylabel('Grid Y')

            for ch in self.channels:
                selected = mask_far * delta[b, ch]
                reg_terms.append(self.norm(selected).mean())
        return torch.stack(reg_terms).mean()


In [ ]:
# @title class::LossScheduler

class LossScheduler:
    def __init__(self,
                 adv_loss_type: str,
                 class_weights: torch.Tensor,
                 lat_grid: torch.Tensor,
                 lon_grid: torch.Tensor,
                 gamma: float = 2.0,
                 reduction: str = 'sum',
                 tversky_alpha: float = 0.9,
                 tversky_beta: float = 0.1,
                 hinge_delta: float = 1.0,
                 topk: int = 50,
                 use_loss_mask = use_loss_mask,
                 loss_mask_ratio = loss_mask_ratio,
                 strict_mask = strict_mask):

        self.adv_loss_type = adv_loss_type
        self.class_weights = class_weights
        self.lat_flat = lat_grid.reshape(-1).to(device)
        self.lon_flat = lon_grid.reshape(-1).to(device)
        self.gamma = gamma
        self.reduction = reduction
        self.tversky_alpha = tversky_alpha
        self.tversky_beta = tversky_beta
        self.hinge_delta = hinge_delta                                          # in degrees

        # self.topk = self.lat_flat.shape[0] * self.lon_flat.shape[0]
        self.topk = topk

        self.use_loss_mask = use_loss_mask
        self.loss_mask_ratio = loss_mask_ratio
        self.strict_mask = strict_mask

    def __call__(self, outputs, targets, p1_prime=None, mask_changed=None):
        loss_mask = self._build_loss_mask(targets) if self.use_loss_mask else None
        if self.adv_loss_type == 'weighted_cross_entropy':
            loss = self.weighted_cross_entropy(outputs, targets, loss_mask=loss_mask)
        elif self.adv_loss_type == 'focal':
            loss = self.focal_loss(outputs, targets, loss_mask=loss_mask, p1_prime=p1_prime, mask_changed=mask_changed)
        elif self.adv_loss_type == 'tversky':
            loss = self.tversky_loss(outputs, targets, loss_mask=loss_mask)
        elif self.adv_loss_type == 'hinge_topk':
            loss = self.hinge_topk_nn_loss(outputs, targets)
        elif self.adv_loss_type == 'weighted_focal':
            loss = self.weighted_focal_loss(outputs, targets, loss_mask=loss_mask)
        elif self.adv_loss_type == 'focal_plus_tversky':
            loss = self.focal_tversky_loss(outputs, targets, loss_mask=loss_mask)
        elif self.adv_loss_type == 'focal_distance_weighted':
            loss = self.focal_loss_dist_weighted(outputs, targets, loss_mask=loss_mask)
        elif self.adv_loss_type == 'cross_entropy':
            loss = self.cross_entropy(outputs, targets, loss_mask=loss_mask)
        else:
            raise ValueError(f"Unknown adv_loss_type: {self.adv_loss_type}")
        return loss.mean()

    def focal_loss(self, outputs, targets, loss_mask=None, p1_prime=None, mask_changed=None):
        if adv_binary_heat_map == True:
            ce = F.cross_entropy(outputs, targets.long().to(outputs.device), reduction='none')
        elif adv_binary_heat_map == False:
            y1 = targets.float().to(outputs.device)
            if y1.dim() == 4 and y1.size(1) == 1:
                y1 = y1[:, 0]
            eps = 1e-8
            tol = 1e-6

            m0 = (y1 <= tol).float()
            m1 = (y1 >= 1.0 - tol).float()
            ms = 1.0 - m0 - m1

            if outputs.dim() == 4 and outputs.size(1) == 2:                     # if the dimension is 4 (i.e., [B,C,H,W]) and the number of classes is 2
                log_probs = F.log_softmax(outputs, dim=1)
                probs = torch.exp(log_probs)
                p0, p1 = probs[:, 0], probs[:, 1]
                log_p0, log_p1 = log_probs[:, 0], log_probs[:, 1]
                ce = -(m0 * log_p0 + m1 * log_p1 + ms * y1 * log_p1)
            else:
                p1 = outputs.clamp(eps, 1.0 - eps).to(y1.device)
                p0 = (1.0 - p1).clamp(eps, 1.0 - eps)
                ce = -(m0 * torch.log(p0) + m1 * torch.log(p1) + ms * y1 * torch.log(p1))

        p_t = torch.exp(-ce)
                                                                                # 'p1_prime': the probability of being classified as 1 (i.e., trajactory node)
        if (p1_prime is not None) and (mask_changed is not None):               # 'mask_changed': (adv_targets.to(self.device) != org_forecasts.to(self.device)).float()
            if targets.dim() == 4 and targets.size(1) == 1:
                y_soft = targets[:, 0].float().to(outputs.device)
            else:
                y_soft = targets.float().to(outputs.device)                     # (B,H,W)

            d = (p1_prime.detach() - y_soft).abs().clamp(0, 1)                  # y_soft: [0,1]

            gmin, gmax = 1.0, 3.0
            gamma_map = gmin + (gmax - gmin) * (1 - d)

            if mask_changed.dim() == 4 and mask_changed.size(1) == 1:
                mc = mask_changed[:, 0].to(outputs.device)
            else:
                mc = mask_changed.to(outputs.device)

            gamma_const = torch.full_like(ce, self.gamma)                       # the shape of 'gamma_const' is the same with that of 'ce', where the values are filled by 'self.gamma'
            g_incorrect = 0.0
            gamma_small = torch.full_like(ce, g_incorrect)
            gamma_var = torch.where(mc.bool(), gamma_small, gamma_const)
        else:
            gamma_var = self.gamma

        focal = (1 - p_t).pow(gamma_var) * ce
        return self._apply_mask_and_reduce(focal, loss_mask)

    def cross_entropy(self, outputs, targets, loss_mask=None):                  # 'output': probs_1
        per_pixel = F.cross_entropy(outputs, targets.long().to(outputs.device), reduction='none')  # (B,H,W)
        return self._apply_mask_and_reduce(per_pixel, loss_mask)  # (B,)

    def weighted_cross_entropy(self, outputs, targets, loss_mask=None):
        w = self.class_weights.to(outputs.device)
        targets_long = targets.long().to(outputs.device)
        ce = F.cross_entropy(outputs, targets_long, reduction='none')           # (B,H,W)
        weight_map = w[targets_long]                                            # (B,H,W)
        wce = ce * weight_map                                                   # (B,H,W)
        return self._apply_mask_and_reduce(wce, loss_mask)

    def focal_loss_dist_weighted(self, outputs, targets, loss_mask=None):
        device = targets.device
        ce = F.cross_entropy(outputs, targets.long().to(device), reduction='none')
        p_t = torch.exp(-ce)
        if targets.dim() == 4:
            targets = targets[:, 0]
        B, H, W = targets.shape

        lats = np.linspace(-90.0, 90.0, H)
        lons = np.linspace(0.0, 360.0, W, endpoint=False)
        lat_vals = torch.tensor(np.deg2rad(lats), dtype=torch.float32, device=device)
        lon_vals = torch.tensor(np.deg2rad(lons), dtype=torch.float32, device=device)
        lat_grid, lon_grid = torch.meshgrid(lat_vals, lon_vals, indexing='ij')
        lat_flat = lat_grid.reshape(-1)
        lon_flat = lon_grid.reshape(-1)

        dist_weight_all = []
        for b in range(B):
            tgt_mask = (targets[b] > 0.5)
            tgt_idx = tgt_mask.view(-1).nonzero(as_tuple=False).squeeze(1)
            if tgt_idx.numel() == 0:
                dist_weight = torch.zeros((H, W), device=device)
            else:
                tgt_lat = lat_flat[tgt_idx]
                tgt_lon = lon_flat[tgt_idx]
                cand_lat = lat_flat.unsqueeze(1)
                cand_lon = lon_flat.unsqueeze(1)
                dlon = cand_lon - tgt_lon
                cosD = cand_lat.sin() * tgt_lat.sin() + cand_lat.cos() * tgt_lat.cos() * dlon.cos()
                D_rad = cosD.clamp(-1, 1).acos()
                min_dist = D_rad.min(dim=1)[0].view(H, W)
                dist_weight = torch.exp(-(min_dist**2) / (2 * sigma**2))
            dist_weight_all.append(dist_weight)

            # plt.figure(figsize=(8, 4))
            # plt.imshow(dist_weight.cpu().numpy(), cmap='jet', origin='lower')
            # target_locs = (targets[b] > 0.5).nonzero(as_tuple=False)
            # if len(target_locs) > 0:
            #     ys = target_locs[:, 0].cpu().numpy()
            #     xs = target_locs[:, 1].cpu().numpy()
            #     plt.scatter(xs, ys, marker='o', s=40)
            # plt.colorbar(label='Min Dist (rad)')
            # plt.title(f'Batch {b}: Min Spherical Distance Map')
            # plt.xlabel('Grid X')
            # plt.ylabel('Grid Y')

        dist_weight_all = torch.stack(dist_weight_all, dim=0)                   # (B,H,W)
        per_pixel = dist_weight_all * ((1 - p_t) ** self.gamma) * ce            # (B,H,W)
        return self._apply_mask_and_reduce(per_pixel, loss_mask)

    def weighted_focal_loss(self, outputs, targets, loss_mask=None):
        ce = F.cross_entropy(outputs, targets.long().to(outputs.device), reduction='none')
        idx = targets.long().to(outputs.device)
        a_t = self.class_weights.to(outputs.device)[idx]
        p_t = torch.exp(-ce)
        focal = a_t * (1 - p_t).pow(self.gamma) * ce
        return self._apply_mask_and_reduce(focal, loss_mask)

    def tversky_loss(self, outputs, targets, loss_mask=None):
        probs = torch.softmax(outputs, dim=1)[:, 1]
        g = targets.to(outputs.device).float()
        p = probs
        if loss_mask is None:
            TP = (p * g).sum(dim=(1, 2))
            FP = (p * (1 - g)).sum(dim=(1, 2))
            FN = ((1 - p) * g).sum(dim=(1, 2))
        else:
            m = loss_mask.to(outputs.device).float()
            TP = (p * g * m).sum(dim=(1, 2))
            FP = (p * (1 - g) * m).sum(dim=(1, 2))
            FN = ((1 - p) * g * m).sum(dim=(1, 2))
        ti = (TP + 1e-6) / (TP + self.tversky_alpha * FP + self.tversky_beta * FN + 1e-6)
        return 1 - ti

    def focal_tversky_loss(self, outputs, targets, loss_mask=None):
        focal_per_sample = self.focal_loss(outputs, targets, loss_mask=loss_mask)
        tversky_per_sample = self.tversky_loss(outputs, targets, loss_mask=loss_mask)

        focal_mean = focal_per_sample.mean().detach()
        tversky_mean = tversky_per_sample.mean().detach()
        print(f"focal_mean: {focal_mean}, tversky_mean: {tversky_mean}")
        total_mean = focal_mean + tversky_mean + 1e-12
        w_focal = tversky_mean / total_mean
        w_tversky = focal_mean / total_mean

        return w_focal * focal_per_sample + w_tversky * tversky_per_sample

    def wce_tversky_loss(self, outputs, targets):
        wce_per_sample = self.weighted_cross_entropy(outputs, targets)          # unmasked path preserved
        tversky_per_sample = self.tversky_loss(outputs, targets)
        wce_mean = wce_per_sample.mean().detach()
        tversky_mean = tversky_per_sample.mean().detach()
        print(f"wce_mean: {wce_mean}, tversky_mean: {tversky_mean}")
        total_mean = wce_mean + tversky_mean + 1e-12
        w_wce = tversky_mean / total_mean
        w_tversky = wce_mean / total_mean
        return w_wce * wce_per_sample + w_tversky * tversky_per_sample

    def _build_loss_mask(self, targets: torch.Tensor) -> torch.Tensor:
        device = targets.device
        tgt_bin = targets if targets.dtype == torch.bool else (targets > 0.5)
        if tgt_bin.dim() == 4:
            tgt_bin = tgt_bin[:, 0]
        B, H, W = tgt_bin.shape

        mask = torch.zeros((B, H, W), dtype=torch.float32, device=device)
        for b in range(B):
            pos = tgt_bin[b]
            neg = ~pos
            neg_idx = torch.nonzero(neg.view(-1), as_tuple=False).squeeze(1)
            n_neg = int(neg_idx.numel())
            if n_neg > 0:
                if self.strict_mask and abs(self.loss_mask_ratio - 0.5) < 1e-8:
                    k = n_neg // 2
                else:
                    k = int(round(self.loss_mask_ratio * n_neg))
                if k > 0:
                    perm = torch.randperm(n_neg, device=device)
                    sel = neg_idx[perm[:k]]
                    neg_sel = torch.zeros(H * W, dtype=torch.bool, device=device)
                    neg_sel[sel] = True
                    neg_sel = neg_sel.view(H, W)
                else:
                    neg_sel = torch.zeros((H, W), dtype=torch.bool, device=device)
            else:
                neg_sel = torch.zeros((H, W), dtype=torch.bool, device=device)

            mask[b] = (pos | neg_sel).float()
        return mask

    def _apply_mask_and_reduce(self, per_pixel: torch.Tensor, loss_mask: torch.Tensor | None) -> torch.Tensor:
        if loss_mask is None:
            if self.reduction == 'mean':
                return per_pixel.mean(dim=(1, 2))
            else:
                return per_pixel.sum(dim=(1, 2))
        else:
            masked = per_pixel * loss_mask
            if self.reduction == 'mean':
                denom = loss_mask.sum(dim=(1, 2)).clamp_min(1.0)
                return masked.sum(dim=(1, 2)) / denom
            else:
                return masked.sum(dim=(1, 2))


In [ ]:
# @title Main (downstream)

_, _, H, W = hat_Z.shape
lats = np.linspace(-90.0,  90.0, H)
lons = np.linspace(0.0, 360.0, W, endpoint=False)
lat_vals = torch.tensor(lats, dtype=torch.float32)                              # [H]
lon_vals = torch.tensor(lons, dtype=torch.float32)                              # [W]
lat_grid, lon_grid = torch.meshgrid(lat_vals, lon_vals, indexing='ij')          # [H, W], [H, W]

def main():
    targets_long = adv_hat_target.long()
    num_classes = int(targets_long.max().item()) + 1

    total = targets_long.numel()
    f_pos = targets_long.sum()
    f_neg = total - f_pos
    w_neg = (1.0 / (f_neg))
    w_pos = (1.0 / (f_pos))
    norm = 2 / (w_neg + w_pos)
    w_neg = w_neg * norm
    w_pos = w_pos * norm
    print(f"w_pos = {w_pos}, w_neg = {w_neg}, w_pos+w_neg={w_pos+w_neg}")
    class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32).to(device)

    loss_scheduler = LossScheduler(pure_adv_loss_type, class_weights, lat_grid, lon_grid)
    reg_scheduler = RegScheduler(adv_reg_type, lat_grid, lon_grid)

    if pure_attack_method == 'AOA':
        attacker = AOA(loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr, beta = 0.5)
    elif pure_attack_method in ['Adam', 'AdamW', 'NAdam', 'Adamax', 'AdaBelief', 'AdaBound']:
        attacker = AdamSeries(loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr)
    elif pure_attack_method in ['PGD', 'TAAOWP']:
        attacker = PGD(loss_scheduler, reg_scheduler, model, epsilon, num_steps, device, lr)
    else:
        raise ValueError(f"Unknown attack_method: {attack_method}")

    pre_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters()}
    N, C, H, W = hat_Z.shape
    mean = mean_tensor.view(1, C, 1, 1).to(device)
    std = std_tensor.view(1, C, 1, 1).to(device)
    best_adv_hat_Z_list, best_adv_forecast_list = [], []

    print(hat_Z.shape)
    num_batches = (N + batch_size - 1) // batch_size
    print(N, batch_size, num_batches)

    for batch_idx, i in enumerate(range(0, N, batch_size), start=1):
        print(f"Batch {batch_idx}/{num_batches}")
        # if batch_idx > 2:
        #     break
        batch_Z = hat_Z[i:i+batch_size].to(device)
        batch_Z_norm = (batch_Z - mean) / std

        batch_org_hat_forecast = org_hat_forecast[i:i+batch_size].to(device)
        batch_org_adv_hat_target = adv_hat_target_no_dilation[i:i+batch_size].to(device)
        batch_adv_hat_target = adv_hat_target[i:i+batch_size].to(device)
        batch_hat_target = hat_target[i:i+batch_size].to(device)

        print(batch_Z_norm.shape, batch_org_hat_forecast.shape, batch_org_adv_hat_target.shape, batch_adv_hat_target.shape, batch_hat_target.shape)

        batch_adv_Z_norm, batch_adv_forecast = attacker(batch_Z_norm, batch_org_hat_forecast, batch_org_adv_hat_target, batch_adv_hat_target, batch_hat_target)

        batch_adv_Z_norm = batch_adv_Z_norm.to(device)
        batch_adv_Z = batch_adv_Z_norm * std + mean
        best_adv_hat_Z_list.append(batch_adv_Z)
        best_adv_forecast_list.append(batch_adv_forecast)

        post_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters()}
        changed = [n for n in pre_state if not torch.equal(pre_state[n], post_state[n])]
        if changed:
            print("Warning: the following model parameters have been altered during attack:")
            for n in changed: print("  ", n)
        else:
            print("Success: no model parameters were modified.")

    adv_forecast = torch.cat(best_adv_forecast_list, dim=0)
    adv_hat_Z = torch.cat(best_adv_hat_Z_list, dim=0)

    if start_iso_list_type == 'readAllDownloaded':
        torch.save(adv_hat_Z, os.path.join(data_dir, f"{prefix}_adv_hat_Z_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))
        if prediction_goal == 'candidate_nodes':
            torch.save(adv_forecast, os.path.join(data_dir, f"{prefix}_adv_hat_O_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))
        elif prediction_goal == 'trajectory':
            torch.save(adv_forecast, os.path.join(data_dir, f"{prefix}_adv_hat_T_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))

    elif start_iso_list_type == 'forCaseStudy':
        torch.save(adv_hat_Z, os.path.join(data_dir, f"{prefix}_adv_hat_Z_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))
        if prediction_goal == 'candidate_nodes':
            torch.save(adv_forecast, os.path.join(data_dir, f"{prefix}_adv_hat_O_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))
        elif prediction_goal == 'trajectory':
            torch.save(adv_forecast, os.path.join(data_dir, f"{prefix}_adv_hat_T_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt"))

if __name__ == "__main__":
    main()


Streaming output truncated to the last 5000 lines.
Step 14: loss = 1.290139, pert_ch0 = 259867.8438, pert_ch2 = 260069.4688, pert_ch3 = 259938.7031, pert_sum = 779876.0000
Step 15: loss = 1.278039, pert_ch0 = 260668.4531, pert_ch2 = 260915.3125, pert_ch3 = 260734.0781, pert_sum = 782317.8750
Step 16: loss = 1.260592, pert_ch0 = 261330.7500, pert_ch2 = 261625.2188, pert_ch3 = 261396.8438, pert_sum = 784352.8125
Step 17: loss = 1.246798, pert_ch0 = 261979.2500, pert_ch2 = 262315.9375, pert_ch3 = 262044.1875, pert_sum = 786339.3750
Step 18: loss = 1.238198, pert_ch0 = 262617.2500, pert_ch2 = 263004.5625, pert_ch3 = 262681.6875, pert_sum = 788303.5000
Step 19: loss = 1.233260, pert_ch0 = 263207.5938, pert_ch2 = 263628.6562, pert_ch3 = 263269.8438, pert_sum = 790106.0625
Step 20: loss = 1.225229, pert_ch0 = 263808.8125, pert_ch2 = 264276.1250, pert_ch3 = 263868.7188, pert_sum = 791953.6875
Step 21: loss = 1.216329, pert_ch0 = 264393.5625, pert_ch2 = 264892.3125, pert_ch3 = 264451.5625, pert

In [ ]:
# @title Convert 'adv_hat_Z' to construct the adversarial '.nc' files for TempestExtremes (downstream)
prefix = 'gt' if gt else 'graphcast'

if start_iso_list_type == 'readAllDownloaded':
    adv_hat_Z_all = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_Z_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")).cpu().numpy()
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_{prediction_goal}.csv"))["dates"].tolist()
elif start_iso_list_type == 'forCaseStudy':
    adv_hat_Z_all = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_Z_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")).cpu().numpy()
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_case_study_{prediction_goal}.csv"))["dates"].tolist()

var_names = ["wind_speed_10m", "elevation", "thickness", "mean_sea_level_pressure"]

count = 0
for start_iso in start_iso_list:
    iso_tag = start_iso.replace(":", "")
    sample_dir = os.path.join(data_dir, iso_tag)

    if prediction_goal == 'candidate_nodes':
        fn_detect = f"adv_{'gt' if gt else 'graphcast'}_detected_nodes_{start_iso.replace(':','')}_{prediction_goal}.txt"
    elif prediction_goal == 'trajectory':
        fn_detect = f"adv_{'gt' if gt else 'graphcast'}_cyclone_tracks_{start_iso.replace(':','')}_{prediction_goal}.txt"
    path_detect = os.path.join(sample_dir, fn_detect)
    if not os.path.exists(path_detect):
        continue

    out_nc = os.path.join(sample_dir, f"adv_{prefix}_tempestextremes_{iso_tag}_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.nc")

    count += 1
    print(count)

    # if os.path.exists(out_nc):
    #     continue

    orig_nc = os.path.join(sample_dir, f"{prefix}_tempestextremes_{iso_tag}.nc")
    ds0 = xr.open_dataset(orig_nc, decode_times=False)
    T = ds0.dims["time"]
    start_time = pd.Timestamp(start_iso)
    dt6 = (start_time + timedelta(hours=6)).strftime("%Y-%m-%dT%H%M")

    i0 = all_dates.index(dt6)
    adv_slice = adv_hat_Z_all[i0 : i0 + T]  # (T, 4, H, W)

    ds = xr.Dataset()
    for c, var in enumerate(var_names):
        arr = adv_slice[:, c, :, :]
        da = xr.DataArray(
            arr,
            dims=("time", "lat", "lon"),
            coords={"time": ds0["time"], "lat": ds0["lat"], "lon": ds0["lon"]},
            name=var
        )
        ds[var] = da
        if "units" in ds0[var].attrs: ds[var].attrs["units"] = ds0[var].attrs["units"]

    ds["time"].encoding.update(ds0["time"].encoding)
    ds["lat"].encoding.update(ds0["lat"].encoding)
    ds["lon"].encoding.update(ds0["lon"].encoding)

    ds.to_netcdf(out_nc)
    ds0.close()
    print(f"Saved {out_nc}")


1


/tmp/ipython-input-1787969126.py:36: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  T = ds0.dims["time"]


Saved /content/drive/My Drive/Climate-data/Graphcast_1deg/2024-12-09T1800/adv_graphcast_tempestextremes_2024-12-09T1800_trajectory_deeplabv3plus_xception_focal_with_heatmap2_AOA_pure_1000_focal_none_10.0_without_heatmap.nc
2


/tmp/ipython-input-1787969126.py:36: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  T = ds0.dims["time"]


Saved /content/drive/My Drive/Climate-data/Graphcast_1deg/2024-11-27T1200/adv_graphcast_tempestextremes_2024-11-27T1200_trajectory_deeplabv3plus_xception_focal_with_heatmap2_AOA_pure_1000_focal_none_10.0_without_heatmap.nc
3


...

/tmp/ipython-input-1787969126.py:36: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  T = ds0.dims["time"]


Saved /content/drive/My Drive/Climate-data/Graphcast_1deg/2019-02-18T1200/adv_graphcast_tempestextremes_2019-02-18T1200_trajectory_deeplabv3plus_xception_focal_with_heatmap2_AOA_pure_1000_focal_none_10.0_without_heatmap.nc
148


/tmp/ipython-input-1787969126.py:36: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  T = ds0.dims["time"]


Saved /content/drive/My Drive/Climate-data/Graphcast_1deg/2019-02-09T0000/adv_graphcast_tempestextremes_2019-02-09T0000_trajectory_deeplabv3plus_xception_focal_with_heatmap2_AOA_pure_1000_focal_none_10.0_without_heatmap.nc
149
Saved /content/drive/My Drive/Climate-data/Graphcast_1deg/2019-02-04T0600/adv_graphcast_tempestextremes_2019-02-04T0600_trajectory_deeplabv3plus_xception_focal_with_heatmap2_AOA_pure_1000_focal_none_10.0_without_heatmap.nc


/tmp/ipython-input-1787969126.py:36: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  T = ds0.dims["time"]


In [ ]:
# @title Generate adversarial candidate nodes and trajectories by TempestExtremes (downstream)

# attack_method = 'Adam'
prefix = 'gt' if gt else 'graphcast'
if start_iso_list_type == 'readAllDownloaded':
    adv_hat_Z_all = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_Z_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")).cpu().numpy()
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_{prediction_goal}.csv"))["dates"].tolist()
elif start_iso_list_type == 'forCaseStudy':
    adv_hat_Z_all = torch.load(os.path.join(data_dir, f"{prefix}_adv_hat_Z_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")).cpu().numpy()
    all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates_case_study_{prediction_goal}.csv"))["dates"].tolist()
all_dates = pd.read_csv(os.path.join(data_dir, f"{prefix}_adv_dates.csv"))["dates"].tolist()

for start_iso in start_iso_list:
    print(start_iso)
    iso_tag = start_iso.replace(":", "")
    sample_dir = os.path.join(data_dir, iso_tag)

    if prediction_goal == 'candidate_nodes':
        fn_detect = f"adv_{'gt' if gt else 'graphcast'}_detected_nodes_{start_iso.replace(':','')}_{prediction_goal}.txt"
    elif prediction_goal == 'trajectory':
        fn_detect = f"adv_{'gt' if gt else 'graphcast'}_cyclone_tracks_{start_iso.replace(':','')}_{prediction_goal}.txt"
    path_detect = os.path.join(sample_dir, fn_detect)
    if not os.path.exists(path_detect):
        continue

    outdir = f"/content/drive/My Drive/Climate-data/Graphcast_1deg/{start_iso.replace(':','')}"

    input_path_for_detect_nodes = os.path.join(outdir, f"adv_{prefix}_tempestextremes_{iso_tag}_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.nc")
    output_path_for_stitch_nodes = os.path.join(outdir, f"adv_{prefix}_detected_nodes_{iso_tag}_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.txt")
    output_path_of_cyclone_trajectories = os.path.join(outdir, f"adv_{prefix}_cyclone_tracks_{iso_tag}_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.txt")

    ds = xr.open_dataset(input_path_for_detect_nodes)
    !chmod +x "/content/drive/My Drive/tempestextremes-master/build/src/nodes/DetectNodes"
    !"/content/drive/My Drive/tempestextremes-master/build/src/nodes/DetectNodes" \
      --in_data "{input_path_for_detect_nodes}" \
      --out "{output_path_for_stitch_nodes}" \
      --searchbymin "mean_sea_level_pressure" \
      --closedcontourcmd "mean_sea_level_pressure,200.0,5.5,0;thickness,-58.8,6.5,1.0" \
      --mergedist 6.0 \
      --outputcmd "mean_sea_level_pressure,min,0;wind_speed_10m,max,2.00000001;elevation,min,0"

    !chmod +x "/content/drive/My Drive/tempestextremes-master/build/src/nodes/StitchNodes"
    !"/content/drive/My Drive/tempestextremes-master/build/src/nodes/StitchNodes" \
      --in "{output_path_for_stitch_nodes}" \
      --out "{output_path_of_cyclone_trajectories}" \
      --in_fmt "lon,lat,slp,wind,zs" \
      --range 8.0 --mintime "54h" \
      --maxgap "24h" \
      --threshold "wind,>=,10.0,10;lat,<=,50.0,10;lat,>=,-50.0,10;zs,<=,150.0,10"


Streaming output truncated to the last 5000 lines.
....Rejected (contour mean_sea_level_pressure): 131
....Rejected (contour thickness): 27
....Done
..Time 2019-11-03 06:00:00
....Total candidates: 19
....Rejected (  location): 0
....Rejected (    merged): 3986
....Rejected ( threshold): 0
....Rejected (contour mean_sea_level_pressure): 122
....Rejected (contour thickness): 27
....Done
..Time 2019-11-03 12:00:00
....Total candidates: 26
....Rejected (  location): 0
....Rejected (    merged): 5202
....Rejected ( threshold): 0
....Rejected (contour mean_sea_level_pressure): 76
....Rejected (contour thickness): 64
....Done
..Time 2019-11-03 18:00:00
....Total candidates: 26
....Rejected (  location): 0
....Rejected (    merged): 4830
....Rejected ( threshold): 0
....Rejected (contour mean_sea_level_pressure): 86
....Rejected (contour thickness): 49
....Done
..Time 2019-11-04 00:00:00
....Total candidates: 47
....Rejected (  location): 0
....Rejected (    merged): 5363
....Rejected ( thres

In [ ]:
# @title Evaluation

import os, sys, torch, numpy as np, pandas as pd
!pip install -q scitools-iris xarray netCDF4 basemap basemap-data-hires scipy
sys.path.append('/content/drive/My Drive/tempest_helper-main')
from tempest_helper.load_trajectories import get_trajectories

var_names = ['wind_speed_10m', 'elevation', 'thickness', 'mean_sea_level_pressure']
mean_tensor = torch.tensor([6.3266e+00, 3.8384e+02, 3.5398e+04, 1.0093e+05]).to(device)
std_tensor = torch.tensor([3.7596,  856.8956, 1829.8486, 1372.3270]).to(device)

try:
    prefix = 'gt' if gt else 'graphcast'
except NameError:
    prefix = 'graphcast'
steps = n_steps - 2

def _lonlat_to_grid(lon, lat, H=181, W=360):
    lon = np.asarray(lon, float)
    lat = np.asarray(lat, float)
    lon = np.where(lon < 0, lon + 360.0, lon)
    gx = np.floor(lon).astype(int).clip(0, W-1)
    gy = np.floor((lat + 90.0) * (H-1) / 180.0).astype(int).clip(0, H-1)
    return gx, gy

def trajs_to_mask(trajs, H=181, W=360):
    Y = np.zeros((H, W), dtype=np.uint8)
    for t in trajs:
        if 'grid_x' in t and 'grid_y' in t:
            gx = np.asarray(t['grid_x']).astype(int)
            gy = np.asarray(t['grid_y']).astype(int)
            gx = np.clip(gx, 0, W-1); gy = np.clip(gy, 0, H-1)
        else:
            gx, gy = _lonlat_to_grid(t['lon'], t['lat'], H=H, W=W)
        Y[gy, gx] = 1
    return torch.from_numpy(Y).to(torch.uint8)

def evaluate_trajectory_pairs_global(data_dir, start_iso_list, prefix, steps, surrogate_model_name, backbone_name, loss_function_type, is_heatmap, attack_method, adv_loss_type, epsilon, adv_is_heatmap, H=181, W=360):
    cols = {"grid_x":0,"grid_y":1,"lon":2,"lat":3,"slp":4,"wind":5,"zs":6,"year":7,"month":8,"day":9,"hour":10}
    TP_sum = TN_sum = FP_sum = FN_sum = 0
    for start_iso in start_iso_list:
        iso_tag = start_iso.replace(":", "")
        sample_dir = os.path.join(data_dir, iso_tag)
        nc_filtered = os.path.join(sample_dir, f"{prefix}_filtered_{iso_tag}.nc")
        target_file = os.path.join(sample_dir, f"adv_{prefix}_cyclone_tracks_{iso_tag}_trajectory.txt")
        adv_traj_file = os.path.join(sample_dir, f"adv_{prefix}_cyclone_tracks_{iso_tag}_trajectory_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.txt")
        if not (os.path.exists(target_file) and os.path.exists(adv_traj_file) and os.path.exists(nc_filtered)):
            continue
        target_trajs = get_trajectories(target_file, nc_filtered, steps, cols)
        adv_trajs = get_trajectories(adv_traj_file, nc_filtered, steps, cols)
        tgt = trajs_to_mask(target_trajs, H=H, W=W).bool()
        pred = trajs_to_mask(adv_trajs, H=H, W=W).bool()
        TP_sum += ((tgt == 1) & (pred == 1)).sum().item()
        TN_sum += ((tgt == 0) & (pred == 0)).sum().item()
        FP_sum += ((tgt == 0) & (pred == 1)).sum().item()
        FN_sum += ((tgt == 1) & (pred == 0)).sum().item()
    denom_neg = TN_sum + FP_sum
    denom_pos = TP_sum + FN_sum
    total = TP_sum + TN_sum + FP_sum + FN_sum
    tn_rate = TN_sum / denom_neg if denom_neg > 0 else 0.0
    fn_rate = FN_sum / denom_pos if denom_pos > 0 else 0.0
    fp_rate = FP_sum / denom_neg if denom_neg > 0 else 0.0
    tp_rate = TP_sum / denom_pos if denom_pos > 0 else 0.0
    acc = (TP_sum + TN_sum) / total if total > 0 else 0.0
    return pd.DataFrame([{
        "TN rate": round(tn_rate, 4),
        "FN rate": round(fn_rate, 4),
        "FP rate": round(fp_rate, 4),
        "TP rate": round(tp_rate, 4),
        "Accuracy": round(acc, 4),
        # "TP": TP_sum, "TN": TN_sum, "FP": FP_sum, "FN": FN_sum, "Total": total
    }])

df_global = evaluate_trajectory_pairs_global(data_dir, start_iso_list, prefix, steps, surrogate_model_name, backbone_name, loss_function_type, is_heatmap, attack_method, adv_loss_type, epsilon, adv_is_heatmap)
print(df_global)

################################### CLOSENESS ##################################
if start_iso_list_type == 'forCaseStudy':                                            # ['readAllDownloaded', 'forCaseStudy']
    adv_path = os.path.join( data_dir, f"{prefix}_adv_hat_Z_case_study_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")
    orig_path = os.path.join( data_dir, f"{prefix}_hat_Z_case_study.pt")
elif start_iso_list_type == 'readAllDownloaded':
    adv_path = os.path.join( data_dir, f"{prefix}_adv_hat_Z_{prediction_goal}_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.pt")
    orig_path = os.path.join( data_dir, f"{prefix}_hat_Z.pt")

x = torch.load(orig_path).float().cpu()
y = torch.load(adv_path).float().cpu()

device = mean_tensor.device
x = x.to(device).float()
y = y.to(device).float()
m = mean_tensor.view(1, -1, 1, 1)
s = std_tensor.view(1, -1, 1, 1).clamp_min(1e-6)
x_n = (x - m) / s
y_n = (y - m) / s
L1_mean = (x_n - y_n).abs().mean()
L1_sum = (x_n - y_n).abs().sum()
print(L1_mean.item())

################################## DETECTION RATE ##############################
def haversine_deg(lon1, lat1, lon2, lat2):
    rlon1 = np.deg2rad(lon1); rlat1 = np.deg2rad(lat1)
    rlon2 = np.deg2rad(lon2); rlat2 = np.deg2rad(lat2)
    dlon = rlon2 - rlon1; dlat = rlat2 - rlat1
    a = np.sin(dlat/2.0)**2 + np.cos(rlat1)*np.cos(rlat2)*np.sin(dlon/2.0)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(np.maximum(1e-15,1-a)))
    return np.rad2deg(c)

def detection_rates_R2(data_dir, start_iso_list, prefix, steps, surrogate_model_name, backbone_name, loss_function_type, is_heatmap, attack_method, adv_loss_type, epsilon, adv_is_heatmap, R):
    cols = {"grid_x":0,"grid_y":1,"lon":2,"lat":3,"slp":4,"wind":5,"zs":6,"year":7,"month":8,"day":9,"hour":10}
    rows = []
    for start_iso in start_iso_list:
        iso_tag = start_iso.replace(":", "")
        sample_dir = os.path.join(data_dir, iso_tag)
        nc_filtered = os.path.join(sample_dir, f"{prefix}_filtered_{iso_tag}.nc")
        target_file = os.path.join(sample_dir, f"adv_{prefix}_cyclone_tracks_{iso_tag}_trajectory.txt")
        adv_traj_file = os.path.join(sample_dir, f"adv_{prefix}_cyclone_tracks_{iso_tag}_trajectory_{surrogate_model_name}_{backbone_name}_{loss_function_type}_{is_heatmap}_{attack_method}_{adv_loss_type}_{epsilon}_{adv_is_heatmap}.txt")
        if not (os.path.exists(target_file) and os.path.exists(adv_traj_file) and os.path.exists(nc_filtered)):
            continue
        target_trajs = get_trajectories(target_file, nc_filtered, steps, cols)
        adv_trajs = get_trajectories(adv_traj_file, nc_filtered, steps, cols)
        tgt_by_t = {}
        for tt in target_trajs:
            lon = np.asarray(tt["lon"], dtype=float); lat = np.asarray(tt["lat"], dtype=float)
            lon = np.where(lon<0, lon+360.0, lon)
            T = min(len(lon), len(lat))
            for t in range(T):
                if t not in tgt_by_t: tgt_by_t[t] = []
                lon_t = lon[t]; lat_t = lat[t]
                lon_t = ((lon_t + 180.0) % 360.0) - 180.0
                tgt_by_t[t].append((lon_t, lat_t))
        for j, ft in enumerate(adv_trajs):
            flon = np.asarray(ft["lon"], dtype=float); flat = np.asarray(ft["lat"], dtype=float)
            flon = np.where(flon<0, flon+360.0, flon)
            T = min(len(flon), len(flat))
            hits = 0
            for t in range(T):
                if t not in tgt_by_t or len(tgt_by_t[t])==0:
                    continue
                lon_f = ((flon[t] + 180.0) % 360.0) - 180.0
                lat_f = flat[t]
                cand = np.array(tgt_by_t[t], dtype=float)
                d = haversine_deg(lon_f, lat_f, cand[:,0], cand[:,1]).min()
                if d < R:
                    hits += 1
            total = T if T>0 else 1
            succ = 100.0 * hits / total
            fail = 100.0 * (1.0 - hits / total)
            rows.append({"iso": iso_tag, "traj_id": j, "success_pct": succ, "failure_pct": fail, "hits": hits, "total": total})
    return pd.DataFrame(rows)

df_det = detection_rates_R2(data_dir, start_iso_list, prefix, steps, surrogate_model_name, backbone_name, loss_function_type, is_heatmap, attack_method, adv_loss_type, epsilon, adv_is_heatmap, R=1.0)
print(df_det[["iso","traj_id","success_pct","failure_pct"]])

total = len(df_det)
success_count = int((df_det["success_pct"] > 50.0).sum())
false_alarm_count = int((df_det["success_pct"] == 0.0).sum())
tc_detection_rate = (success_count / total) if total > 0 else 0.0
tc_false_alarm_rate = (false_alarm_count / total) if total > 0 else 0.0
print({
    "total_trajs": total,
    "success_count": success_count,
    "false_alarm_count": false_alarm_count,
    "TC_detection_rate": tc_detection_rate,
    "TC_false_alarm_rate": tc_false_alarm_rate
})


/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:341: IrisCfMissingVarWarning: Missing CF-netCDF auxiliary coordinate variable 'expver', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:341: IrisCfMissingVarWarning: Missing CF-netCDF auxiliary coordinate variable 'number', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:759: IrisCfMissingVarWarning: Missing CF-netCDF label variable 'expver', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:759: IrisCfMissingVarWarning: Missing CF-netCDF label variable 'number', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/common/mixin.py:206: FutureWarning: You are using legacy date precision for Iris units - 

   TN rate  FN rate  FP rate  TP rate  Accuracy
0   0.9999   0.7971   0.0001   0.2029    0.9998
0.04498394951224327


/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:341: IrisCfMissingVarWarning: Missing CF-netCDF auxiliary coordinate variable 'expver', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:341: IrisCfMissingVarWarning: Missing CF-netCDF auxiliary coordinate variable 'number', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:759: IrisCfMissingVarWarning: Missing CF-netCDF label variable 'expver', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/fileformats/cf.py:759: IrisCfMissingVarWarning: Missing CF-netCDF label variable 'number', referenced by netCDF variable 'mean_sea_level_pressure'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/iris/common/mixin.py:206: FutureWarning: You are using legacy date precision for Iris units - 

                 iso  traj_id  success_pct  failure_pct
0    2024-11-27T1200        0    50.000000    50.000000
1    2024-11-14T1200        0    75.000000    25.000000
2    2024-11-03T0600        0    16.666667    83.333333
3    2024-10-25T0000        0    33.333333    66.666667
4    2024-10-25T0000        1     0.000000   100.000000
..               ...      ...          ...          ...
126  2019-04-25T1800        0    30.000000    70.000000
127  2019-03-18T0000        0    58.333333    41.666667
128  2019-03-11T1800        0    54.545455    45.454545
129  2019-02-18T1200        0    30.000000    70.000000
130  2019-02-04T0600        0    27.272727    72.727273

[131 rows x 4 columns]
{'total_trajs': 131, 'success_count': 39, 'false_alarm_count': 20, 'TC_detection_rate': 0.29770992366412213, 'TC_false_alarm_rate': 0.15267175572519084}
